# Генерация Терры

Модель `ldm_v4` дообучена с двумя условиями, оба можно задать при генерации:

- **кто** (`WHO`): `'terra'`, `'other'` (любая другая кошка) или `None` (условие «пусто»);
- **тип** (`TYPE`): кластер признаков DINOv2, `None`, `'terra'` (основной кластер Терры) или номер 0-99.
  Кластеры в основном про окрас и породу. Терра в кластере длинношерстных колорпойнтов вместе с ~330 похожими кошками.
  Как выглядит каждый тип, показывает ячейка «Типы кошек», все сразу: `review/clusters_*.jpg` в папке датасета.

При генерации работает classifier-free guidance (CFG): выход модели с условиями отодвигается от выхода без условий
с силой `W`. Больше `W` = больше похоже на Терру, но меньше разнообразие и выше риск пересыщенных цветов.
Поверх можно включить autoguidance слабой моделью (`GUIDE`).

30 фото Терры отложены и в обучении не участвовали (`terra_holdout.json`). Последняя ячейка показывает
для каждой картинки ближайшее обучающее фото и отмечает копии.

**Какой снимок брать** (`runs/eval/terra_v4/`, 512 сэмплов «Терра + тип»; у отложенных настоящих фото копий 3.3%):

| Снимок | w | Похожа на Терру | Копии |
|---|---|---|---|
| 5 000 | 2 | 97.7% | 0.6% |
| **7 500** | **2** | **98.8%** | **2.5%** |
| 7 500 | 3 | 100% | 9.2% |
| 10 000 | 2 | 99.6% | 15.6% |
| 25 000 (`ckpt/last.ckpt`) | 2 | 99.8% | 60.4% |

Дальше 7 500 модель все сильнее повторяет обучающие фото, поэтому по умолчанию `ema_0007500.pt`.

Сэмплер: `METHOD = 'sde'` (стохастический, 250 шагов) или `'heun'` (ОДУ, 50 шагов, детерминированный: один шум = одна картинка).
На моделях без условий Heun давал размазанные картинки; ячейка «SDE и ОДУ» сравнивает их на одних и тех же шумах.

In [ ]:
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import AutoencoderKL
from PIL import Image, ImageOps

from dataset import read_manifest
from guidance import CFG, Guided
from models import UNet
from train import VPSDE, sample

RUN_DIR = Path('/home/CCP/semenchuk.aa/DiffusionLearn/runs/ldm_v4')
DATA = Path('/home/CCP/semenchuk.aa/DiffusionLearn_data/pretrain_v1')
DEVICE = 'cuda'

cfg = json.loads((RUN_DIR / 'config.json').read_text())
WHO_NULL, K = cfg['cond_sizes']        # значения «пусто» у условий: кто = 2, тип = 100
TERRA_TYPE = cfg['terra_type']         # основной кластер Терры
print(f"шагов {cfg['steps']}, Терра {cfg['terra_frac']:.0%} примеров, кластеров {K}, тип Терры {TERRA_TYPE}, scale {cfg['scale']:.5f}")

vae = AutoencoderKL.from_pretrained(cfg['vae'], local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)
sde = VPSDE(pred=cfg['pred'])
bf16 = lambda: torch.autocast('cuda', dtype=torch.bfloat16)


def load_ema(path):
    """EMA-веса из ckpt/last.ckpt или ema_*.pt любого запуска, конфигурация из его config.json. Возвращает модель и шаг."""
    path = Path(path)
    run = path.parent.parent if path.parent.name == 'ckpt' else path.parent
    c = json.loads((run / 'config.json').read_text())
    ck = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    sd = ck['ema'] if 'ema' in ck else {k[4:]: v for k, v in ck['state_dict'].items() if k.startswith('ema.')}
    m = UNet(img_size=64, in_ch=4, base=c['base'], mults=[int(v) for v in c['mults'].split(',')], num_res=c['num_res'],
             attn_res=[int(v) for v in c['attn_res'].split(',')], dropout=c['dropout'], norm_groups=32,
             head_ch=c.get('head_ch') or None, qk_norm=c.get('qk_norm', False), cond_sizes=c.get('cond_sizes', []))
    m.load_state_dict(sd)
    return m.to(DEVICE).eval().requires_grad_(False), ck.get('step', ck.get('global_step'))


@torch.no_grad()
def decode(z, chunk=8):
    """Латенты модели (B, 4, 64, 64) -> картинки uint8 (B, 3, 512, 512) на CPU."""
    out = []
    for i in range(0, len(z), chunk):
        x = vae.decode(z[i:i + chunk].to(DEVICE).float() / cfg['scale']).sample
        out.append(((x.clamp(-1, 1) + 1) * 127.5).round().byte().cpu())
    return torch.cat(out)


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) для imshow."""
    return img.permute(1, 2, 0).numpy()


@torch.no_grad()
def generate(model, n, seed=0, who='terra', type_=None, w=2.0, method='sde', steps=None, guide=None, w_ag=1.5, chunk=16):
    """
    n картинок с условиями. Возвращает латенты (n, 4, 64, 64) на CPU.
      who      : 'terra', 'other' (любая другая кошка) или None (условие «пусто»)
      type_    : None («пусто»), 'terra' (основной кластер Терры) или номер кластера 0..K-1
      w        : сила CFG от ответа без условий, 1 = без CFG. Если оба условия пустые, w ни на что не влияет
      method   : 'sde' (Эйлер-Маруяма) или 'heun' (ОДУ)
      steps    : по умолчанию 250 для SDE и 50 для Heun
      guide    : слабая модель для autoguidance с теми же условиями, w_ag ее сила
    Один и тот же seed дает одни и те же стартовые шумы при любых настройках, поэтому варианты можно сравнивать.
    """
    steps = steps or (250 if method == 'sde' else 50)
    y = torch.tensor([{'terra': 1, 'other': 0, None: WHO_NULL}[who],
                      K if type_ is None else TERRA_TYPE if type_ == 'terra' else int(type_)])
    y0 = torch.tensor([WHO_NULL, K])
    net = CFG(model, y, y0, w)
    if guide is not None:
        net = Guided(net, CFG(guide, y, y0, w), w_ag)
    x0 = torch.randn(n, 4, 64, 64, generator=torch.Generator().manual_seed(seed))
    out = []
    for i in range(0, n, chunk):
        torch.manual_seed(seed * 1_000_003 + i)                     # шум SDE тоже повторяется
        out.append(sample(net, sde, x0[i:i + chunk].to(DEVICE), steps=steps, rho=1.0 if method == 'sde' else 2.0,
                          method=method, amp_ctx=bf16, clip=False).cpu())
    return torch.cat(out)


def show(rows, labels, title, ncol):
    """Сетка картинок: rows = список батчей uint8, строка = вариант."""
    fig, axs = plt.subplots(len(rows), ncol, figsize=(2.8 * ncol, 2.9 * len(rows)), squeeze=False)
    for r, ims in enumerate(rows):
        for c in range(ncol):
            axs[r, c].imshow(to_np(ims[c]))
            axs[r, c].set_xticks([])
            axs[r, c].set_yticks([])
        axs[r, 0].set_ylabel(labels[r], fontsize=12)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

## Типы кошек

Случайные фото других кошек из одного кластера: так видно, что значит номер `TYPE`.

In [ ]:
SHOW_TYPE = TERRA_TYPE       # номер кластера 0..99

assign = json.loads((DATA / 'clusters.json').read_text())['assign']
members = [p for p, c in assign.items() if c == SHOW_TYPE and not p.startswith('terra/')]
pick = random.Random(0).sample(members, min(8, len(members)))
fig, axs = plt.subplots(1, len(pick), figsize=(2.6 * len(pick), 2.9), squeeze=False)
for ax, p in zip(axs[0], pick):
    ax.imshow(ImageOps.fit(ImageOps.exif_transpose(Image.open(DATA / p)).convert('RGB'), (256, 256)))
    ax.axis('off')
plt.suptitle(f'тип {SHOW_TYPE}: {len(members)} кадров других кошек' + (', основной тип Терры' if SHOW_TYPE == TERRA_TYPE else ''))
plt.tight_layout()
plt.show()

## Генерация

Картинки сохраняются в `runs/ldm_v4/generated_terra/<имя>/`, имя собирается из параметров.

In [ ]:
# ------------------------------------------------------------- параметры
CKPT = 'ema_0007500.pt'      # лучший баланс похожести и новизны; 'ckpt/last.ckpt' (25 000) в основном копирует
WHO = 'terra'                # 'terra', 'other' или None
TYPE = None                  # None, 'terra' (основной кластер Терры) или номер кластера 0..99
W = 2.0                      # сила CFG: 1 = без нее, 2 лучше всего по метрикам, 3 уже чаще копирует
METHOD = 'sde'               # 'sde' или 'heun' (ОДУ)
STEPS = None                 # None = 250 для SDE, 50 для Heun
GUIDE = None                 # autoguidance слабой моделью
# GUIDE = '/home/CCP/semenchuk.aa/DiffusionLearn/runs/ldm_v3/ckpt/last.ckpt'   # ldm_v3: та же модель без условий и без Терры
W_AG = 1.5
SEED = 1
BATCH = 16
COLS = 4

# ------------------------------------------------------------- генерация
model, step = load_ema(RUN_DIR / CKPT)
guide = load_ema(GUIDE)[0] if GUIDE else None
lat = generate(model, BATCH, seed=SEED, who=WHO, type_=TYPE, w=W, method=METHOD, steps=STEPS, guide=guide, w_ag=W_AG)
imgs = decode(lat)
del guide
torch.cuda.empty_cache()

name = (f'step{step:07d}_{WHO or "any"}' + (f'_type{TYPE}' if TYPE is not None else '') + f'_w{W}_{METHOD}{STEPS or ""}'
        + (f'_ag-{Path(GUIDE).parts[-3]}-w{W_AG}' if GUIDE else '') + f'_seed{SEED}')
out_dir = RUN_DIR / 'generated_terra' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, im in enumerate(imgs):
    Image.fromarray(to_np(im)).save(out_dir / f'{i:03d}.png')
print(f'{BATCH} картинок сохранено в {out_dir}')

rows = math.ceil(BATCH / COLS)
fig, axs = plt.subplots(rows, COLS, figsize=(4.5 * COLS, 4.7 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, im in enumerate(imgs):
    axs.flat[i].imshow(to_np(im))
    axs.flat[i].set_title(str(i))
plt.suptitle(name, fontsize=15)
plt.tight_layout()
plt.show()

## Сила CFG

Одни и те же шумы и условия из ячейки генерации, строка = `W`. Видно, как растет похожесть и падает разнообразие.

In [ ]:
W_LIST = [1.0, 1.5, 2.0, 3.0, 4.0]
N_CMP = 6

rows = [decode(generate(model, N_CMP, seed=SEED, who=WHO, type_=TYPE, w=w, method=METHOD, steps=STEPS)) for w in W_LIST]
show(rows, [f'W = {w}' for w in W_LIST], f'step {step}, кто {WHO}, тип {TYPE}, {METHOD}', N_CMP)

## SDE и ОДУ

Одни и те же стартовые шумы и условия. SDE на каждом шаге добавляет свежий шум и этим исправляет ошибки модели,
ОДУ (Heun) идет детерминированно и обычно дает более гладкие, но и более размытые картинки.

In [ ]:
N_CMP = 6
VARIANTS = [('sde', 250), ('sde', 1000), ('heun', 50), ('heun', 200)]

rows = [decode(generate(model, N_CMP, seed=SEED, who=WHO, type_=TYPE, w=W, method=m, steps=st)) for m, st in VARIANTS]
show(rows, [f'{m} {st}' for m, st in VARIANTS], f'step {step}, кто {WHO}, тип {TYPE}, W {W}', N_CMP)

## Новая это Терра или копия

Как в `tools/eval_terra.py`: каждое обучающее фото Терры раскладывается на кропы по правилу обучения
(бокс внутри, несколько размеров и сдвигов, отражение) в 64x64, и для картинки ищется ближайший кроп.
d1 = расстояние до ближайшего фото, d2 = до второго ближайшего другого фото, `ratio = d1 / d2`.
Копия близка к одному фото намного сильнее, чем к остальным: `ratio < 0.7`, подпись красная.
У 30 отложенных настоящих фото Терры так выходит около 3%.

Первый запуск ячейки около минуты: строятся кропы 254 обучающих фото.

In [ ]:
from tools.eval_terra import copy_ratio, copy_refs

N_SHOW = 8

if 'refs' not in globals():
    hold = set(json.loads((DATA / 'terra_holdout.json').read_text()))
    train_rows = [r for r in read_manifest(DATA / 'manifest.jsonl') if r['source'] == 'terra' and r['path'] not in hold]
    refs, owner = copy_refs(DATA, train_rows)

x = imgs[:N_SHOW].float() / 127.5 - 1
ratio, nn = copy_ratio(x, refs, owner, len(train_rows))
ratio, nn = ratio.cpu(), nn.cpu()
print(f'ratio по картинкам: ' + ', '.join(f'{v:.2f}' for v in ratio.tolist())
      + f'; копий {(ratio < 0.7).sum().item()} из {N_SHOW}')

fig, axs = plt.subplots(math.ceil(N_SHOW / 2), 4, figsize=(16, 4.2 * math.ceil(N_SHOW / 2)), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for j in range(N_SHOW):
    r, c = divmod(j, 2)
    real = ImageOps.fit(ImageOps.exif_transpose(Image.open(DATA / train_rows[nn[j]]['path'])).convert('RGB'), (512, 512))
    axs[r, 2 * c].imshow(to_np(imgs[j]))
    axs[r, 2 * c].set_title(f'{j}: ratio {ratio[j]:.2f}' + (', копия' if ratio[j] < 0.7 else ''),
                            color='red' if ratio[j] < 0.7 else 'black')
    axs[r, 2 * c + 1].imshow(real)
    axs[r, 2 * c + 1].set_title('ближайшее обучающее фото')
plt.tight_layout()
plt.show()